# Assignment 01 Runner (Modular + Magic Commands)

This notebook is now a thin orchestrator.
- Core logic is implemented in `assignment_01_pipeline.py`
- The module is loaded with notebook magic (`%run`)
- Run cells in order

In [1]:
# Optional dependency install
%pip install -q openai pandas tqdm pydantic openpyxl

Note: you may need to restart the kernel to use updated packages.


In [1]:
# Load all modular functions/classes into notebook scope using magic command
%run ./ecom_llm.py
%run ./llm_judge.py

In [ ]:
import os
os.environ["NEBIUS_API_KEY"] = "Your_key"

In [7]:
# Preconditions:
# 1) Place your dataset at: data/Assignment_01_product_dataset.csv
# 2) CSV columns must include: product_name, Product_attribute_list, material, warranty
# 3) Environment variable NEBIUS_API_KEY is set

data_path = "data/ecommerce_dataset.csv"

results_df, full_df, summary_overview, summary_distribution = run_assignment(
    df=None,
    excel_path="ecommerce_sheet_new.xlsx",
    data_path=data_path,
)

print("Saved and updated: ecommerce_sheet_new.xlsx")
print(f"Loaded dataset from: {data_path}")
print("results_df (trimmed output columns):")
display(results_df.head())
print("full_df (includes explanation columns):")
display(full_df.head())

Task 2: Generating:   0%|          | 0/50 [00:00<?, ?it/s]

Task 5: Judging:   0%|          | 0/50 [00:00<?, ?it/s]

Saved and updated: ecommerce_sheet_new.xlsx
Loaded dataset from: data/ecommerce_dataset.csv
results_df (trimmed output columns):


,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,tone,length,grounding,latency_rating,cost_rating,final_score
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,"Enjoy a seamless, responsive experience with t...",1165,768,98,ok,good,ok,good,bad,good,good,fail
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Capture every detail with the incredible 200MP...,918,768,96,ok,good,ok,good,good,good,good,pass
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,Imagine capturing stunning photos with a 50MP ...,715,783,80,ok,good,ok,good,ok,good,good,pass
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty,Enjoy immersive sound experiences with the Son...,826,765,104,good,good,good,ok,good,good,good,pass
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty,Imagine enjoying your music without any distra...,808,754,86,ok,good,ok,ok,good,good,good,pass


full_df (includes explanation columns):


,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,...,length,grounding,latency_rating,cost_rating,final_score,fluency_explanation,grammar_explanation,tone_explanation,length_explanation,grounding_explanation
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,"Enjoy a seamless, responsive experience with t...",1165,768,98,ok,good,...,good,bad,good,good,fail,"The description reads naturally, but there are...",There are no spelling or punctuation errors in...,The description uses some corporate filler wor...,I counted 76 words in the description.,Step A: Extract required facts from SOURCE DAT...
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Capture every detail with the incredible 200MP...,918,768,96,ok,good,...,good,good,good,good,pass,"The description reads naturally, but there are...",There are no spelling or punctuation errors in...,"The description uses some GOOD signals, such a...",I counted 76 words in the description.,Step A: Extract required facts from SOURCE DAT...
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,Imagine capturing stunning photos with a 50MP ...,715,783,80,ok,good,...,good,ok,good,good,pass,"The description reads naturally, with a clear ...",There are no spelling or punctuation errors in...,The tone of the description is conversational ...,I counted 69 words in the description.,Step A: Extract required facts from SOURCE DAT...
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty,Enjoy immersive sound experiences with the Son...,826,765,104,good,good,...,ok,good,good,good,pass,"The description reads naturally, with no awkwa...",There are no spelling or punctuation errors in...,The description has a good tone. It addresses ...,I counted 76 words in the description.,Step A: Extract required facts from SOURCE DAT...
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty,Imagine enjoying your music without any distra...,808,754,86,ok,good,...,ok,good,good,good,pass,"The description reads naturally, but there are...",There are no spelling or punctuation errors in...,"The description uses some GOOD signals, such a...",I counted 76 words in the description.,Step A: Extract required facts from SOURCE DAT...


In [4]:
# Recovery cell: run only the last example and append to previously saved dataframes
import pandas as pd

from llm_client import get_client
from llm_generator import run_generation
from llm_judge import run_judge
from llm_scoring import build_summary
from ecom_llm import load_input_df

PREV_RESULTS_PATH = "assignment_01_new.xlsx"
PREV_FULL_PATH = "assignment_01_full_new.xlsx"
TMP_LAST_GEN_PATH = "_tmp_last_gen.xlsx"
TMP_LAST_JUDGE_PATH = "_tmp_last_judge.xlsx"

# Load previous successful outputs
prev_results_df = pd.read_excel(PREV_RESULTS_PATH)
prev_full_df = pd.read_excel(PREV_FULL_PATH)

# Run pipeline only on the last input row
input_df = load_input_df(data_path)
last_input_df = input_df.tail(1).copy()

client = get_client()
last_generated_df = run_generation(client=client, df=last_input_df, excel_path=TMP_LAST_GEN_PATH)
last_results_df, last_full_df = run_judge(
    results_df=last_generated_df.copy(),
    client=client,
    excel_path=TMP_LAST_JUDGE_PATH,
)

# Append last example to previous outputs, keeping the latest version per product row key
key_cols = ["product_name", "Product_attribute_list", "material", "warranty"]

results_df = pd.concat([prev_results_df, last_results_df], ignore_index=True)
results_df = results_df.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)

full_df = pd.concat([prev_full_df, last_full_df], ignore_index=True)
full_df = full_df.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)

summary_overview, summary_distribution = build_summary(results_df)

# Persist refreshed outputs for the rest of the notebook
results_df.to_excel(PREV_RESULTS_PATH, index=False)
full_df.to_excel(PREV_FULL_PATH, index=False)

print("Appended last example to previous outputs and refreshed in-memory dataframes.")
print(f"results_df rows: {len(results_df)} | full_df rows: {len(full_df)}")
display(results_df.tail(3))

Task 2: Generating:   0%|          | 0/1 [00:00<?, ?it/s]

Task 5: Judging:   0%|          | 0/1 [00:00<?, ?it/s]

Appended last example to previous outputs and refreshed in-memory dataframes.
results_df rows: 50 | full_df rows: 50


,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,tone,length,grounding,latency_rating,cost_rating,final_score,cost
47,Acer Predator XB273U 27″ Monitor,"features: 2560×1440 170 Hz IPS, HDR400, G‑Sync...",plastic & metal stand,3‑year warranty,Immerse yourself in the action with the Acer P...,986,471,110,ok,good,ok,ok,good,good,good,pass,0.000016
48,BenQ PD2725U 27″ Monitor,"features: 4K UHD IPS, 98% P3, Thunderbolt 3; c...",metal stand,3‑year warranty,Enjoy stunning clarity and vibrant colors with...,878,460,96,good,good,good,good,good,good,good,pass,0.000015
49,ASUS ProArt PA278QV Monitor,"features: 27″ 2560×1440 IPS, 100% sRGB, Calman...",metal stand,3‑year warranty,"Enjoy vibrant, accurate colors on the 27″ ASUS...",1254,562,126,good,good,ok,ok,good,good,good,pass,NaN


In [8]:
# Add cost column per row to both dataframes
# $0.02 / 1M In
# $0.06 / 1M Out (with caching)
results_df["cost"] = results_df.apply(
    lambda row: 0.02 * row["input_tokens"] / 1e6 + 0.06 * row["output_tokens"] / 1e6,
    axis=1
)
full_df["cost"] = full_df.apply(
    lambda row: 0.02 * row["input_tokens"] / 1e6 + 0.06 * row["output_tokens"] / 1e6,
    axis=1
)

In [9]:
results_df.to_excel("assignment_01_new.xlsx", index=False)
full_df.to_excel("assignment_01_full_new.xlsx", index=False)
print("Saved: assignment_01_new.xlsx and assignment_01_full_new.xlsx")

Saved: assignment_01_new.xlsx and assignment_01_full_new.xlsx


In [10]:
rows = results_df.sample(n=15, random_state=42).copy()
rows

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,tone,length,grounding,latency_rating,cost_rating,final_score,cost
13,Nintendo Switch OLED,"features: 7″ OLED screen, docked & handheld mo...",plastic chassis,1‑year limited warranty,Enjoy vibrant colors and stunning visuals with...,656,757,82,ok,good,ok,bad,bad,good,good,fail,0.000020
39,Blink Outdoor 4,"features: 1080p HD, 2‑year battery, weather‑re...",plastic,1‑year limited warranty,Enjoy crystal-clear 1080p HD video footage of ...,672,756,79,good,good,ok,good,good,good,good,pass,0.000020
30,LEGO Star Wars Millennium Falcon 75192,"features: 7 541 pieces, detailed interior, min...",ABS plastic bricks,lifetime replacement of missing bricks,"Imagine building the iconic Millennium Falcon,...",1017,764,86,ok,good,ok,good,good,good,good,pass,0.000020
45,SanDisk Extreme PRO 128 GB SDXC,"features: UHS‑I U3, 200 MB/s read, waterproof;...",plastic,lifetime limited warranty,You'll enjoy capturing every moment with the S...,800,765,98,good,good,ok,ok,good,good,good,pass,0.000021
17,Keurig K‑Supreme Plus Smart,"features: multistream extraction, BrewID, 78 o...",brushed stainless steel accents,1‑year limited warranty,"Enjoy perfectly brewed coffee, every time, wit...",811,759,84,good,good,good,good,good,good,good,pass,0.000020
48,BenQ PD2725U 27″ Monitor,"features: 4K UHD IPS, 98% P3, Thunderbolt 3; c...",metal stand,3‑year warranty,Want to experience your digital content in stu...,745,766,95,good,good,good,good,good,good,good,pass,0.000021
26,Adidas Ultraboost Light,"features: Light BOOST midsole, Primeknit+ uppe...",textile & synthetic,2‑year manufacturer warranty,Imagine running on clouds – that’s the feeling...,867,778,92,ok,good,ok,ok,good,good,good,pass,0.000021
25,Nike Air Zoom Pegasus 40,"features: React foam, Zoom Air units, engineer...",textile & synthetic,2‑year manufacturer warranty,Experience the effortless energy return of the...,762,754,99,good,good,ok,ok,good,good,good,pass,0.000021
32,Kindle Paperwhite 11th Gen,"features: 6.8″ glare‑free display, 10‑week bat...",recycled plastic,1‑year limited warranty,Enjoy weeks of reading on a single charge with...,881,765,106,good,good,good,ok,good,good,good,pass,0.000022
19,Dyson V15 Detect Vacuum,"features: laser dust detection, HEPA filtratio...",polycarbonate & aluminum,2‑year parts & labor,Experience the power of Dyson's laser dust det...,1025,759,118,ok,good,ok,bad,good,good,good,fail,0.000022


In [17]:
import re

def count_words(text: str) -> int:
    # Strip punctuation artifacts, split on whitespace
    return len(re.findall(r'\b\w+\b', text))

def classify_length(word_count: int) -> str:
    if 50 <= word_count <= 90:
        return "good"
    elif (40 <= word_count <= 49) or (91 <= word_count <= 110):
        return "ok"
    else:
        return "bad"

In [12]:
rows["generated_description"] = rows["generated_description"].str.replace("\n", "<br>", regex=False)
from IPython.display import HTML, display
display(HTML(rows[["product_name","generated_description", "Product_attribute_list"]].to_html(escape=False, index=False)))

product_name,generated_description,Product_attribute_list
Nintendo Switch OLED,"Enjoy vibrant colors and stunning visuals with the Nintendo Switch OLED's 7” OLED screen. Seamlessly transition between docked and handheld modes for ultimate gaming flexibility. With a large 64GB storage capacity, you'll have plenty of room for all your favorite games. The Nintendo Switch OLED, housed in a durable plastic chassis, comes with a 1-year limited warranty.","features: 7″ OLED screen, docked & handheld modes, 64 GB storage; capacity: large"
Blink Outdoor 4,"Enjoy crystal-clear 1080p HD video footage of your home, day or night, with the Blink Outdoor 4. Its weather-resistant plastic design can withstand the elements, while the 2‑year battery life keeps it running reliably. This limited edition Blink Outdoor 4 comes with a 1‑year limited warranty for your added peace of mind.","features: 1080p HD, 2‑year battery, weather‑resistant; limited edition"
LEGO Star Wars Millennium Falcon 75192,"Imagine building the iconic Millennium Falcon, brick by brick, with its detailed interior and all your favorite minifigs. The LEGO Star Wars Millennium Falcon 75192 lets you do just that with 7,541 pieces for a truly immersive build. Enjoy hours of creative fun knowing that your build is powered by long-lasting batteries and that LEGO will replace any missing bricks for a lifetime.","features: 7 541 pieces, detailed interior, minifigs included; battery: long‑lasting"
SanDisk Extreme PRO 128 GB SDXC,"You'll enjoy capturing every moment with the SanDisk Extreme PRO 128GB SDXC. This card features UHS-I U3 technology and a 200MB/s read speed, so you can shoot high-quality video and photos without lag. Plus, its waterproof design means you can take it anywhere, knowing your adventures are protected. With a long-lasting battery and a lifetime limited warranty, you can focus on what matters most.","features: UHS‑I U3, 200 MB/s read, waterproof; battery: long‑lasting"
Keurig K‑Supreme Plus Smart,"Enjoy perfectly brewed coffee, every time, with the Keurig K‑Supreme Plus Smart. Its multistream extraction technology and BrewID feature work together to meet your individual coffee preference. Plus, with a large 78oz reservoir and brushed stainless steel accents, this machine is as stylish as it is convenient. Backed by a 1-year limited warranty, you can brew with confidence.","features: multistream extraction, BrewID, 78 oz reservoir; capacity: large"
BenQ PD2725U 27″ Monitor,"Want to experience your digital content in stunning clarity? The BenQ PD2725U 27″ Monitor delivers vibrant colors and sharp details with its 4K UHD IPS display and 98% P3 wide color gamut. Enjoy seamless connectivity with the Thunderbolt 3 port, and appreciate the sleek, modern design of the metal stand. Your creative projects will truly shine with this reliable monitor backed by a 3‑year warranty.","features: 4K UHD IPS, 98% P3, Thunderbolt 3; color options: multiple"
Adidas Ultraboost Light,"Imagine running on clouds – that’s the feeling you get with the Adidas Ultraboost Light. The Light BOOST midsole and Primeknit+ upper work together to keep your feet light and comfortable mile after mile, while the Continental rubber outsole provides superior grip on any surface. With a 4.7/5 rating from runners who love these shoes, you can feel confident you’re choosing a top-quality running experience.","features: Light BOOST midsole, Primeknit+ upper, Continental rubber; rating: 4.7/5"
Nike Air Zoom Pegasus 40,"Experience the effortless energy return of the Nike Air Zoom Pegasus 40. These shoes feature React foam and Zoom Air units for a smooth, responsive ride. The engineered mesh upper keeps your feet cool and comfortable, mile after mile. Designed to be energy efficient, the Pegasus 40 will help you push further and feel your best on every run. They're backed by a 2-year manufacturer warranty so you can enjoy your new shoes with confidence.","features: React foam, Zoom Air units, engineered mesh upp

In [15]:
# Choose random rows from results_df
import numpy as np
# Fluency:
# 1  The Nintendo Switch OLED, housed in a -- Clunky phrasing
# 6 and appreciate the sleek, modern design of the metal stand. Your creative projects will truly shine -- awkward phrasing
# 10 You'll see dust you never knew existed, and the V15 Detect's HEPA filtration will capture it -- awkward phrasing
# 13 he TP‑Link Deco XE75 Mesh WiFi 6E. This tri‑band AXE5400 system, with a 3‑pack covering up to 7,200 sq ft, -- unnatural wording, ok probabillistic model
rows['fluency_human'] = np.array(["ok", "good", "good", "good", "good", "ok", "good", "good", "good", "ok", "good", "good", "ok", "good", "good"])

# Grammar:
# At least to me, seems good
rows['grammar_human'] = np.array(["good", "good", "good", "good", "good", "good", "good", "good", "good", "good", "good", "good", "good", "good", "good"])

# Tone:
# 1 Seamlessly transition between docked and handheld modes for ultimate gaming --buzz words
# 2 withstand the elements is a nice touch. gave it good
# 3 sounds friendly but this is just... too much Enjoy hours of creative fun knowing that your build is powered by long-lasting batteries
# 4 Plus, its waterproof design means you can take it anywhere, knowing your adventures are protected -- corporate filler 
# 6 appreciate the sleek, modern design of the metal stand. Your creative projects will truly shine -- awkward and clunky, ok
# 7 a top-quality running experience -- a bit awkward but the rest is impressive so it's good overall
# 10 trust in the Dyson V15 Detect's long-lasting performance. -- corporate filler
# 11 elevate your filmmaking adventures. -- buzz words
# 13 Enjoy seamless WiFi coverag smart choice - buzz words

rows['tone_human'] = np.array(["ok", "good", "ok", "ok", "good", "ok", "good", "good", "good", "ok", "ok", "good", "ok", "good", "good"])

# Grounding:
# 10 You'll see dust you never knew existed, and the V15 Detect's HEPA filtration will capture it -- you dont really see the dust
# Overall Seems good, harder to judge (my llm judge seems to do great job there)
rows['grounding_human'] = np.array(["good", "good", "good", "good", "good", "good", "good", " good", "good", "ok", "good", "good", "good", "good", "good"])
rows['fluency_match'] = rows['fluency_human'] == rows['fluency']
rows['grammar_match'] = rows['grammar_human'] == rows['grammar']
rows['tone_match'] = rows['tone_human'] == rows['tone']
rows['grounding_match'] = rows['grounding_human'].str.strip() == rows['grounding'].str.strip()
print("Fluency matches:", rows['fluency_match'].sum(), "/", len(rows))
print("Grammar matches:", rows['grammar_match'].sum(), "/", len(rows))
print("Tone matches:", rows['tone_match'].sum(), "/", len(rows))
print("Grounding matches:", rows['grounding_match'].sum(), "/", len(rows))

Fluency matches: 11 / 15
Grammar matches: 15 / 15
Tone matches: 10 / 15
Grounding matches: 13 / 15


*Seems like tone matches the least, in grounding we had a weird generation that the LLM did not catch. Might need to update the prompt or change the models.*

In [18]:
import pandas as pd

rows["length_human"] = rows["generated_description"].apply(count_words)
rows["true_length_label"] = rows["length_human"].apply(classify_length)
rows["judge_correct"] = rows["length"] == rows["true_length_label"]

# Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report
print(classification_report(rows["true_length_label"], rows["length"]))

              precision    recall  f1-score   support

         bad       0.00      0.00      0.00         0
        good       1.00      0.27      0.42        15
          ok       0.00      0.00      0.00         0

    accuracy                           0.27        15
   macro avg       0.33      0.09      0.14        15
weighted avg       1.00      0.27      0.42        15



c:\Users\maorb\anaconda3\envs\prez\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\maorb\anaconda3\envs\prez\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\maorb\anaconda3\envs\prez\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


*The length was off which is unexpected since it's raw counts. Looking at the judge length_explanation, we might need to edit the prompt or change the model*

In [19]:
rows.to_excel("assignment_01_sample_rows_new.xlsx", index=False)
print("Saved: assignment_01_sample_rows_new.xlsx")

Saved: assignment_01_sample_rows_new.xlsx


In [20]:
print("=== Final Summary ===")
display(summary_overview)
display(summary_distribution)

=== Final Summary ===


,metric,value
0,pass_rate_percent,92.0
1,pass_count,46.0
2,total_count,50.0


,good,ok,bad,total
fluency,34,16,0,50
grammar,50,0,0,50
tone,28,22,0,50
length,12,35,3,50
grounding,46,2,2,50
latency_rating,50,0,0,50
cost_rating,50,0,0,50
